# GPT-2 source-level textual unpredictability: Poetry

This Kaggle notebook follows the existing PDF layout reconstruction and stanza-preserving chunking pipeline. It scores each poem chunk with **GPT-2 only**, converts the chunk-level bits-per-character (BPC) scores into capped frequency histograms, and calculates the mean, variance, skewness, and excess kurtosis for each poet from the binned frequencies.

Chunks accumulate complete stanzas until reaching at least 200 words; no stanza is split. A short poem remains one chunk, and an undersized final remainder of a longer poem is merged into the preceding chunk. Moments are frequency-weighted estimates from histogram bin centers, using Freedman–Diaconis binning capped at 8–30 bins.

Enable Internet in Kaggle to download GPT-2 if it is not already cached.

In [1]:
# Install dependencies used by the PDF parser and GPT-2 scorer.
%pip install -q pymupdf transformers accelerate scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 72.1 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import os
import re
import math
import json
import random
import traceback
import hashlib
import zipfile
from collections import Counter

import fitz
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import display

DATASET_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/poet_gpt2_moments")
EXTRACT_ROOT = Path("/kaggle/working/poet_dataset_extracted")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

MIN_WORDS = 200
MODEL_NAME = "gpt2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
STRIDE = 512
LOWERCASE_FOR_LM = False
HISTOGRAM_METHOD = "fd"
MIN_BINS = 8
MAX_BINS = 30

HEADER_FRACTION = 0.085
FOOTER_FRACTION = 0.075
STANZA_GAP_MULTIPLIER = 1.55
TITLE_UPPERCASE_RATIO = 0.82
TITLE_MAX_CHARS = 120
TITLE_MAX_WORDS = 15
TITLE_FONT_FACTOR = 1.18

NON_POEM_PATTERNS = [
    r"^source note$", r"^poems recovered", r"^working corpus",
    r"^acknowledg(?:e)?ments?$", r"^about the author$", r"^copyright",
    r"^contents$", r"^table of contents$", r"^introduction$",
    r"^foreword$", r"^notes? on the text$", r"^books by ",
    r"^also by ", r"^publisher", r"^penguin books", r"^gallery books", r"^faber",
]

TARGET_POETS = [
    "Arthur Sze", "Tracy K. Smith", "Marie Howe", "Margaret Atwood",
    "Ocean Vuong", "Rita Dove", "Kaveh Akbar", "Don Paterson",
    "A. E. Stallings", "Maria Hummel", "Patricia Lockwood", "Megan Fernandes",
    "Amanda Gorman", "Eleanor Wilner", "Sasha Debevec-McKenney", "Nick Flynn",
    "Aria Aber", "Evie Shockley", "John Lee Clark", "Kimiko Hahn", "Vona Groarke",
    "Danez Smith", "Max Ritvo", "Anselm Berrigan", "Meena Alexander",
    "Lizzie Harris", "Andrea Cohen", "Nick Laird", "Hala Alyan", "Terrance Hayes",
    "Jennifer Chang", "Frieda Hughes",
]

zip_files = sorted(DATASET_ROOT.rglob("*.zip"))
for zip_path in zip_files:
    dest = EXTRACT_ROOT / zip_path.stem
    dest.mkdir(parents=True, exist_ok=True)
    marker = dest / ".extracted"
    if not marker.exists():
        with zipfile.ZipFile(zip_path, "r") as archive:
            archive.extractall(dest)
        marker.write_text("ok", encoding="utf-8")

print("Dataset:", DATASET_ROOT)
print("Output:", OUTPUT_ROOT)
print("Model:", MODEL_NAME)
print("Device:", DEVICE)
print("Minimum chunk size:", MIN_WORDS, "words")

Dataset: /kaggle/input
Output: /kaggle/working/poet_gpt2_moments
Model: gpt2
Device: cuda
Minimum chunk size: 200 words


In [3]:
import os
import re
import math
import json
import random
import traceback
from collections import Counter

import fitz
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModelForCausalLM

print("PyMuPDF loaded.")
print("Transformers loaded.")

PyMuPDF loaded.
Transformers loaded.


## 1. Find poet PDFs

In [4]:
def normalize_name(s):
    s = s.lower().replace("_", " ").replace("-", " ")
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

normalized_poets = {
    normalize_name(p): p
    for p in TARGET_POETS
}

def identify_poet(path):
    stem = normalize_name(path.stem)
    matches = [
        poet
        for norm, poet in normalized_poets.items()
        if norm in stem
    ]
    return matches[0] if len(matches) == 1 else None

pdf_candidates = set(DATASET_ROOT.rglob("*.pdf")) | set(EXTRACT_ROOT.rglob("*.pdf"))
pdf_files = sorted((p.resolve() for p in pdf_candidates), key=lambda p: str(p))

discovery_rows = []
poet_to_paths = {}

for path in pdf_files:
    poet = identify_poet(path)

    discovery_rows.append({
        "file": str(path),
        "poet": poet,
        "recognized": poet is not None,
    })

    if poet is not None:
        poet_to_paths.setdefault(poet, []).append(path)

discovery_df = pd.DataFrame(discovery_rows)

print("PDFs discovered:", len(pdf_files))
print(
    "Recognized target-poet PDFs:",
    int(discovery_df["recognized"].sum())
)

for poet in TARGET_POETS:
    paths = poet_to_paths.get(poet, [])
    print(f"{poet:30s} -> {len(paths)} file(s)")
    for p in paths:
        print("   ", p)

discovery_df.to_csv(
    OUTPUT_ROOT / "discovery_report.csv",
    index=False,
    encoding="utf-8-sig",
)

PDFs discovered: 32
Recognized target-poet PDFs: 31
Arthur Sze                     -> 1 file(s)
    /kaggle/input/datasets/srabongitikar/poetries/andrea_cohen_32_poets_delivery/01_arthur_sze.pdf
Tracy K. Smith                 -> 1 file(s)
    /kaggle/input/datasets/srabongitikar/poetries/andrea_cohen_32_poets_delivery/02_tracy_k_smith.pdf
Marie Howe                     -> 1 file(s)
    /kaggle/input/datasets/srabongitikar/poetries/andrea_cohen_32_poets_delivery/03_marie_howe.pdf
Margaret Atwood                -> 1 file(s)
    /kaggle/input/datasets/srabongitikar/poetries/andrea_cohen_32_poets_delivery/04_margaret_atwood.pdf
Ocean Vuong                    -> 1 file(s)
    /kaggle/input/datasets/srabongitikar/poetries/andrea_cohen_32_poets_delivery/05_ocean_vuong.pdf
Rita Dove                      -> 1 file(s)
    /kaggle/input/datasets/srabongitikar/poetries/andrea_cohen_32_poets_delivery/06_rita_dove.pdf
Kaveh Akbar                    -> 1 file(s)
    /kaggle/input/datasets/srabongitik

## 2. Reconstruct poems and stanzas from PDF layout

In [5]:
compiled_non_poem = [
    re.compile(pattern, re.I)
    for pattern in NON_POEM_PATTERNS
]

def normalize_spaces(text):
    return re.sub(r"[ \t]+", " ", text.strip())

def is_roman_label(text):
    return bool(
        re.fullmatch(r"[IVXLCDM]+", text.strip(), flags=re.I)
    )

def is_non_poem_line(text):
    t = normalize_spaces(text)

    if not t:
        return True

    if is_roman_label(t):
        return True

    if re.fullmatch(r"\d{1,4}", t):
        return True

    return any(rx.search(t) for rx in compiled_non_poem)

def uppercase_ratio(text):
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return 0.0
    return sum(c.isupper() for c in letters) / len(letters)

def count_words(text):
    return len(re.findall(
        r"(?u)\b[\wÀ-ÖØ-öø-ÿ]+(?:['’\-][\wÀ-ÖØ-öø-ÿ]+)*\b",
        text
    ))

def extract_page_lines(page, page_number):
    data = page.get_text("dict")
    lines = []

    for block in data.get("blocks", []):
        if block.get("type") != 0:
            continue

        for line in block.get("lines", []):
            spans = [
                sp
                for sp in line.get("spans", [])
                if sp.get("text", "").strip()
            ]

            if not spans:
                continue

            spans.sort(key=lambda sp: sp["bbox"][0])

            text = "".join(
                sp["text"]
                for sp in spans
            ).strip()

            if not text:
                continue

            x0 = min(sp["bbox"][0] for sp in spans)
            y0 = min(sp["bbox"][1] for sp in spans)
            x1 = max(sp["bbox"][2] for sp in spans)
            y1 = max(sp["bbox"][3] for sp in spans)

            font_size = float(np.median([
                float(sp.get("size", 10.0))
                for sp in spans
            ]))

            lines.append({
                "page": page_number,
                "text": text,
                "x0": x0,
                "y0": y0,
                "x1": x1,
                "y1": y1,
                "font_size": font_size,
                "page_height": page.rect.height,
                "page_width": page.rect.width,
            })

    lines.sort(
        key=lambda x: (x["y0"], x["x0"])
    )

    return lines

def looks_like_title(
    text,
    font_size,
    typical_body_font,
):
    t = normalize_spaces(text)

    if not t or is_non_poem_line(t):
        return False

    if len(t) > TITLE_MAX_CHARS:
        return False

    if count_words(t) > TITLE_MAX_WORDS:
        return False

    if sum(c.isalpha() for c in t) < 2:
        return False

    if uppercase_ratio(t) >= TITLE_UPPERCASE_RATIO:
        return True

    if (
        typical_body_font
        and font_size
        >= TITLE_FONT_FACTOR * typical_body_font
    ):
        return True

    return False

def build_poems(pdf_path):
    doc = fitz.open(str(pdf_path))

    all_pages = []
    body_sizes = []

    for page_number, page in enumerate(doc, start=1):
        lines = extract_page_lines(
            page,
            page_number,
        )

        kept = []

        for line in lines:
            relative_y = (
                line["y0"]
                / max(line["page_height"], 1)
            )

            if relative_y < HEADER_FRACTION:
                continue

            if relative_y > 1.0 - FOOTER_FRACTION:
                continue

            if is_non_poem_line(line["text"]):
                continue

            kept.append(line)

            if (
                count_words(line["text"]) > 0
                and uppercase_ratio(line["text"]) < 0.95
            ):
                body_sizes.append(line["font_size"])

        all_pages.append(kept)

    doc.close()

    typical_body_font = (
        float(np.median(body_sizes))
        if body_sizes
        else 10.0
    )

    events = []

    for page_lines in all_pages:
        gaps = [
            b["y0"] - a["y1"]
            for a, b
            in zip(page_lines, page_lines[1:])
            if b["y0"] >= a["y1"]
        ]

        base_gap = (
            float(np.median(gaps))
            if gaps
            else 4.0
        )

        for line in page_lines:
            events.append({
                "kind": (
                    "title"
                    if looks_like_title(
                        line["text"],
                        line["font_size"],
                        typical_body_font,
                    )
                    else "body"
                ),
                "line": line,
                "base_gap": base_gap,
            })

    poems = []
    current = None

    def finalize():
        nonlocal current

        if current is None:
            return

        if current["current_stanza"]:
            current["stanzas"].append(
                current["current_stanza"]
            )
            current["current_stanza"] = []

        if current["stanzas"]:
            poems.append({
                "title": current["title"],
                "start_page": current["start_page"],
                "end_page": current["end_page"],
                "stanzas": current["stanzas"],
            })

        current = None

    for event in events:
        line = event["line"]

        if event["kind"] == "title":
            finalize()

            current = {
                "title": normalize_spaces(
                    line["text"]
                ),
                "start_page": line["page"],
                "end_page": line["page"],
                "stanzas": [],
                "current_stanza": [],
                "base_gap": event["base_gap"],
                "previous_line": None,
            }

            continue

        if current is None:
            continue

        stanza_break = False
        previous = current["previous_line"]

        if (
            previous is not None
            and previous["page"] == line["page"]
        ):
            gap = line["y0"] - previous["y1"]

            if gap > (
                current["base_gap"]
                * STANZA_GAP_MULTIPLIER
            ):
                stanza_break = True

        if stanza_break and current["current_stanza"]:
            current["stanzas"].append(
                current["current_stanza"]
            )
            current["current_stanza"] = []

        current["current_stanza"].append(line)
        current["end_page"] = line["page"]
        current["previous_line"] = line

    finalize()

    return poems

## 3. Create stanza-preserving chunks

In [6]:
def stanza_text(stanza):
    return "\n".join(
        line["text"].rstrip()
        for line in stanza
    )

def create_chunks(poems, min_words):
    chunks = []

    for poem_index, poem in enumerate(
        poems,
        start=1,
    ):
        stanza_rows = []

        for stanza_index, stanza in enumerate(
            poem["stanzas"],
            start=1,
        ):
            text = stanza_text(stanza)

            stanza_rows.append({
                "stanza_index": stanza_index,
                "text": text,
                "word_count": count_words(text),
                "page_start": stanza[0]["page"],
                "page_end": stanza[-1]["page"],
            })

        if not stanza_rows:
            continue

        poem_word_count = sum(
            row["word_count"]
            for row in stanza_rows
        )

        # Entire short poem = one chunk.
        if poem_word_count < min_words:
            chunks.append({
                "poem_index": poem_index,
                "chunk_within_poem": 1,
                "poem_title": poem["title"],
                "poem_start_page": poem["start_page"],
                "poem_end_page": poem["end_page"],
                "stanza_start": 1,
                "stanza_end": len(stanza_rows),
                "stanza_count": len(stanza_rows),
                "word_count": poem_word_count,
                "text": "\n\n".join(
                    row["text"]
                    for row in stanza_rows
                ),
            })
            continue

        current = []
        current_words = 0
        poem_chunks = []

        for stanza in stanza_rows:
            current.append(stanza)
            current_words += stanza["word_count"]

            if current_words >= min_words:
                poem_chunks.append({
                    "poem_index": poem_index,
                    "chunk_within_poem": (
                        len(poem_chunks) + 1
                    ),
                    "poem_title": poem["title"],
                    "poem_start_page": poem["start_page"],
                    "poem_end_page": poem["end_page"],
                    "stanza_start": (
                        current[0]["stanza_index"]
                    ),
                    "stanza_end": (
                        current[-1]["stanza_index"]
                    ),
                    "stanza_count": len(current),
                    "word_count": current_words,
                    "text": "\n\n".join(
                        row["text"]
                        for row in current
                    ),
                })

                current = []
                current_words = 0

        # Do not create an undersized final chunk for a long poem.
        if current:
            remainder_text = "\n\n".join(
                row["text"]
                for row in current
            )

            if (
                poem_chunks
                and current_words < min_words
            ):
                previous = poem_chunks[-1]

                previous["stanza_end"] = (
                    current[-1]["stanza_index"]
                )
                previous["stanza_count"] += len(current)
                previous["word_count"] += current_words
                previous["text"] += (
                    "\n\n" + remainder_text
                )
            else:
                poem_chunks.append({
                    "poem_index": poem_index,
                    "chunk_within_poem": (
                        len(poem_chunks) + 1
                    ),
                    "poem_title": poem["title"],
                    "poem_start_page": poem["start_page"],
                    "poem_end_page": poem["end_page"],
                    "stanza_start": (
                        current[0]["stanza_index"]
                    ),
                    "stanza_end": (
                        current[-1]["stanza_index"]
                    ),
                    "stanza_count": len(current),
                    "word_count": current_words,
                    "text": remainder_text,
                })

        chunks.extend(poem_chunks)

    for chunk_id, chunk in enumerate(
        chunks,
        start=1,
    ):
        chunk["chunk_id"] = chunk_id

    return chunks

## 4. Parse all available poets

In [7]:
poet_chunks = {}
source_status = []

for poet in TARGET_POETS:
    matched_paths = poet_to_paths.get(poet, [])
    if not matched_paths:
        source_status.append({"source_name": poet, "source_pdf": "", "status": "NO_PDF_FOUND", "n_poems": 0, "n_chunks": 0, "error": ""})
        print(f"[MISSING] {poet}: no matching PDF")
        continue

    pdf_path = matched_paths[0]
    try:
        poems = build_poems(pdf_path)
        chunks = create_chunks(poems, MIN_WORDS)
        if not chunks:
            raise RuntimeError("No chunks were created by the poetry parser.")
        poet_chunks[poet] = {"pdf_path": str(pdf_path), "poems": poems, "chunks": chunks}
        source_status.append({"source_name": poet, "source_pdf": str(pdf_path), "status": "READY", "n_poems": len(poems), "n_chunks": len(chunks), "error": ""})
        print(f"{poet}: {len(poems)} poems, {len(chunks)} chunks")
        if len(matched_paths) > 1:
            print(f"  Multiple files matched; using {pdf_path.name}")
    except Exception as exc:
        source_status.append({"source_name": poet, "source_pdf": str(pdf_path), "status": "PARSER_ERROR", "n_poems": 0, "n_chunks": 0, "error": f"{type(exc).__name__}: {exc}"})
        print(f"[ERROR] {poet}: {type(exc).__name__}: {exc}")

source_status_df = pd.DataFrame(source_status)
source_status_df.to_csv(OUTPUT_ROOT / "poet_run_status_gpt2.csv", index=False, encoding="utf-8-sig")
print("Sources ready:", len(poet_chunks), "of", len(TARGET_POETS))

Arthur Sze: 1 poems, 1 chunks
Tracy K. Smith: 126 poems, 146 chunks
Marie Howe: 32 poems, 34 chunks
Margaret Atwood: 17 poems, 217 chunks
Ocean Vuong: 26 poems, 33 chunks
Rita Dove: 110 poems, 115 chunks
Kaveh Akbar: 59 poems, 61 chunks
Don Paterson: 81 poems, 107 chunks
[MISSING] A. E. Stallings: no matching PDF
Maria Hummel: 32 poems, 33 chunks
Patricia Lockwood: 24 poems, 41 chunks
Megan Fernandes: 58 poems, 58 chunks
Amanda Gorman: 129 poems, 146 chunks
Eleanor Wilner: 110 poems, 132 chunks
Sasha Debevec-McKenney: 39 poems, 54 chunks
Nick Flynn: 1 poems, 1 chunks
Aria Aber: 90 poems, 103 chunks
Evie Shockley: 27 poems, 59 chunks
John Lee Clark: 43 poems, 46 chunks
Kimiko Hahn: 122 poems, 162 chunks
Vona Groarke: 49 poems, 56 chunks
Danez Smith: 15 poems, 58 chunks
Max Ritvo: 33 poems, 33 chunks
Anselm Berrigan: 19 poems, 34 chunks
Meena Alexander: 39 poems, 46 chunks
Lizzie Harris: 1 poems, 1 chunks
Andrea Cohen: 69 poems, 70 chunks
Nick Laird: 27 poems, 36 chunks
Hala Alyan: 41 po

## 5. Load GPT-2 and define next-token scoring

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

possible_lengths = [
    getattr(tokenizer, "model_max_length", None),
    getattr(model.config, "n_positions", None),
    getattr(model.config, "max_position_embeddings", None),
]
valid_lengths = [x for x in possible_lengths if isinstance(x, int) and 0 < x < 100000]
MODEL_CONTEXT = min(valid_lengths) if valid_lengths else 1024

print("Loaded:", MODEL_NAME)
print("Context length:", MODEL_CONTEXT)
print("Device:", DEVICE)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded: gpt2
Context length: 1024
Device: cuda


In [9]:
def prepare_lm_text(text):
    return (
        text.lower()
        if LOWERCASE_FOR_LM
        else text
    )

def score_text_lm(text):
    text_for_model = prepare_lm_text(
        text
    )

    encoded = tokenizer(
        text_for_model,
        return_tensors="pt",
        add_special_tokens=False,
    )

    input_ids = encoded[
        "input_ids"
    ]

    sequence_length = (
        input_ids.size(1)
    )

    if sequence_length < 2:
        return {
            "total_bits": np.nan,
            "scored_tokens": 0,
            "bits_per_token": np.nan,
            "bpc": np.nan,
            "perplexity": np.nan,
            "model_tokens": sequence_length,
            "character_count": len(
                text_for_model
            ),
        }

    total_nll_nats = 0.0
    scored_tokens = 0

    previous_end = 1

    with torch.no_grad():
        for begin in range(
            0,
            sequence_length - 1,
            STRIDE,
        ):
            end = min(
                begin + MODEL_CONTEXT,
                sequence_length,
            )

            window = input_ids[
                :,
                begin:end,
            ].to(DEVICE)

            outputs = model(
                input_ids=window
            )

            logits = outputs.logits

            shift_logits = logits[
                :,
                :-1,
                :,
            ]

            shift_labels = window[
                :,
                1:,
            ]

            absolute_target_start = max(
                previous_end,
                begin + 1,
            )

            relative_target_start = (
                absolute_target_start
                - begin
            )

            if (
                relative_target_start
                > shift_labels.size(1)
            ):
                continue

            shifted_start = (
                relative_target_start - 1
            )

            selected_logits = (
                shift_logits[
                    :,
                    shifted_start:,
                    :,
                ]
            )

            selected_labels = (
                shift_labels[
                    :,
                    shifted_start:,
                ]
            )

            log_probs = torch.log_softmax(
                selected_logits,
                dim=-1,
            )

            target_log_probs = (
                log_probs.gather(
                    2,
                    selected_labels.unsqueeze(
                        -1
                    ),
                ).squeeze(-1)
            )

            total_nll_nats += float(
                -target_log_probs.sum().item()
            )

            scored_here = int(
                selected_labels.numel()
            )

            scored_tokens += (
                scored_here
            )

            previous_end = end

            if end >= sequence_length:
                break

    total_bits = (
        total_nll_nats
        / math.log(2)
    )

    bits_per_token = (
        total_bits / scored_tokens
        if scored_tokens
        else np.nan
    )

    mean_nll_nats = (
        total_nll_nats
        / scored_tokens
        if scored_tokens
        else np.nan
    )

    perplexity = (
        math.exp(mean_nll_nats)
        if (
            np.isfinite(
                mean_nll_nats
            )
            and mean_nll_nats < 50
        )
        else np.inf
    )

    chars = len(
        text_for_model
    )

    bpc = (
        total_bits / chars
        if chars
        else np.nan
    )

    return {
        "total_bits": total_bits,
        "scored_tokens": scored_tokens,
        "bits_per_token": bits_per_token,
        "bpc": bpc,
        "perplexity": perplexity,
        "model_tokens": sequence_length,
        "character_count": chars,
    }

## 6. Score chunks, bin BPC frequencies, and export source moments

In [10]:
def frequency_histogram(values, max_bins=MAX_BINS, min_bins=MIN_BINS):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        raise ValueError("No finite BPC values to bin.")

    if values.size == 1 or np.isclose(values.min(), values.max()):
        center = float(values[0])
        edges = np.array([center - 0.5, center + 0.5], dtype=float)
    else:
        fd_edges = np.histogram_bin_edges(values, bins="fd")
        proposed_bins = max(1, len(fd_edges) - 1)
        n_bins = int(np.clip(proposed_bins, min_bins, max_bins))
        edges = np.linspace(float(values.min()), float(values.max()), n_bins + 1)

    counts, edges = np.histogram(values, bins=edges)
    centers = (edges[:-1] + edges[1:]) / 2
    keep = counts > 0
    return pd.DataFrame({
        "bin_left": edges[:-1][keep],
        "bin_right": edges[1:][keep],
        "bin_center": centers[keep],
        "frequency": counts[keep].astype(int),
    })


def moments_from_frequencies(hist):
    centers = hist["bin_center"].to_numpy(dtype=float)
    frequencies = hist["frequency"].to_numpy(dtype=float)
    n = float(frequencies.sum())
    if n <= 0:
        raise ValueError("Histogram has no observations.")

    mean = float(np.sum(frequencies * centers) / n)
    centered = centers - mean
    m2 = float(np.sum(frequencies * centered**2) / n)
    m3 = float(np.sum(frequencies * centered**3) / n)
    m4 = float(np.sum(frequencies * centered**4) / n)

    if m2 > 0:
        skewness = m3 / (m2 ** 1.5)
        excess_kurtosis = m4 / (m2 ** 2) - 3.0
    else:
        skewness = np.nan
        excess_kurtosis = np.nan

    return {
        "n_valid_chunks": int(n),
        "n_histogram_bins": int(len(hist)),
        "mean_bpc": mean,
        "variance_bpc": m2,
        "skewness_bpc": float(skewness) if np.isfinite(skewness) else np.nan,
        "excess_kurtosis_bpc": float(excess_kurtosis) if np.isfinite(excess_kurtosis) else np.nan,
    }

In [11]:
chunk_rows = []
frequency_rows = []
moment_rows = []

for poet, payload in poet_chunks.items():
    print(f"\nScoring {poet} ...")
    valid_scores = []
    for chunk in payload["chunks"]:
        score = score_text_lm(chunk["text"])
        row = {
            "source_name": poet,
            "source_pdf": payload["pdf_path"],
            "chunk_id": chunk["chunk_id"],
            "poem_index": chunk["poem_index"],
            "poem_title": chunk["poem_title"],
            "stanza_start": chunk["stanza_start"],
            "stanza_end": chunk["stanza_end"],
            "stanza_count": chunk["stanza_count"],
            "word_count": chunk["word_count"],
            "model_name": MODEL_NAME,
            "bpc": score["bpc"],
            "bits_per_token": score["bits_per_token"],
            "scored_tokens": score["scored_tokens"],
            "character_count": score["character_count"],
        }
        chunk_rows.append(row)
        if np.isfinite(score["bpc"]):
            valid_scores.append(float(score["bpc"]))

    source_chunks_df = pd.DataFrame([r for r in chunk_rows if r["source_name"] == poet])
    if not valid_scores:
        moment_rows.append({
            "source_name": poet, "source_pdf": payload["pdf_path"],
            "model_name": MODEL_NAME, "status": "NO_VALID_CHUNKS",
            "n_poems": len(payload["poems"]), "n_chunks": len(payload["chunks"]),
            "n_valid_chunks": 0, "n_histogram_bins": 0,
            "mean_bpc": np.nan, "variance_bpc": np.nan,
            "skewness_bpc": np.nan, "excess_kurtosis_bpc": np.nan,
        })
        continue

    hist = frequency_histogram(valid_scores)
    hist.insert(0, "source_name", poet)
    hist.insert(1, "model_name", MODEL_NAME)
    frequency_rows.extend(hist.to_dict("records"))
    stats = moments_from_frequencies(hist)
    moment_rows.append({
        "source_name": poet,
        "source_pdf": payload["pdf_path"],
        "model_name": MODEL_NAME,
        "status": "SUCCESS",
        "n_poems": len(payload["poems"]),
        "n_chunks": len(payload["chunks"]),
        **stats,
    })
    print(f"  valid chunks={len(valid_scores)}, mean BPC≈{stats['mean_bpc']:.5f}")

# Retain parser failures and missing sources in the main source-level CSV.
existing = {r["source_name"] for r in moment_rows}
for row in source_status:
    if row["source_name"] not in existing:
        moment_rows.append({
            "source_name": row["source_name"], "source_pdf": row["source_pdf"],
            "model_name": MODEL_NAME, "status": row["status"],
            "n_poems": row["n_poems"], "n_chunks": row["n_chunks"],
            "n_valid_chunks": 0, "n_histogram_bins": 0,
            "mean_bpc": np.nan, "variance_bpc": np.nan,
            "skewness_bpc": np.nan, "excess_kurtosis_bpc": np.nan,
            "error": row.get("error", ""),
        })

poet_moments_df = pd.DataFrame(moment_rows).sort_values("source_name").reset_index(drop=True)
poet_chunks_df = pd.DataFrame(chunk_rows)
poet_frequencies_df = pd.DataFrame(frequency_rows)

moments_path = OUTPUT_ROOT / "poet_source_moments_gpt2.csv"
chunks_path = OUTPUT_ROOT / "poet_chunk_scores_gpt2.csv"
frequencies_path = OUTPUT_ROOT / "poet_binned_frequencies_gpt2.csv"

poet_moments_df.to_csv(moments_path, index=False, encoding="utf-8-sig")
poet_chunks_df.to_csv(chunks_path, index=False, encoding="utf-8-sig")
poet_frequencies_df.to_csv(frequencies_path, index=False, encoding="utf-8-sig")

print("\nMain output:", moments_path)
display(poet_moments_df)


Scoring Arthur Sze ...
  valid chunks=1, mean BPC≈1.71963

Scoring Tracy K. Smith ...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1128 > 1024). Running this sequence through the model will result in indexing errors


  valid chunks=146, mean BPC≈1.70360

Scoring Marie Howe ...
  valid chunks=34, mean BPC≈1.49371

Scoring Margaret Atwood ...
  valid chunks=217, mean BPC≈1.68601

Scoring Ocean Vuong ...
  valid chunks=33, mean BPC≈1.97108

Scoring Rita Dove ...
  valid chunks=115, mean BPC≈1.99675

Scoring Kaveh Akbar ...
  valid chunks=61, mean BPC≈1.75192

Scoring Don Paterson ...
  valid chunks=107, mean BPC≈2.14111

Scoring Maria Hummel ...
  valid chunks=33, mean BPC≈1.73944

Scoring Patricia Lockwood ...
  valid chunks=41, mean BPC≈1.55334

Scoring Megan Fernandes ...
  valid chunks=58, mean BPC≈2.23236

Scoring Amanda Gorman ...
  valid chunks=144, mean BPC≈3.17054

Scoring Eleanor Wilner ...
  valid chunks=132, mean BPC≈1.71432

Scoring Sasha Debevec-McKenney ...
  valid chunks=54, mean BPC≈1.50302

Scoring Nick Flynn ...
  valid chunks=1, mean BPC≈1.71963

Scoring Aria Aber ...
  valid chunks=103, mean BPC≈3.02890

Scoring Evie Shockley ...
  valid chunks=57, mean BPC≈2.25518

Scoring John L

,source_name,source_pdf,model_name,status,n_poems,n_chunks,n_valid_chunks,n_histogram_bins,mean_bpc,variance_bpc,skewness_bpc,excess_kurtosis_bpc,error
0,A. E. Stallings,,gpt2,NO_PDF_FOUND,0,0,0,0,NaN,NaN,NaN,NaN,
1,Amanda Gorman,/kaggle/input/datasets/srabongitikar/poetries/...,gpt2,SUCCESS,129,146,144,8,3.170542,3.759763,0.984687,-0.686090,NaN
2,Andrea Cohen,/kaggle/input/datasets/srabongitikar/poetries/...,gpt2,SUCCESS,69,70,70,8,2.187264,0.252283,0.641693,-0.782562,NaN
3,Anselm Berrigan,/kaggle/input/datasets/srabongitikar/poetries/...,gpt2,SUCCESS,19,34,34,7,2.149159,0.040661,0.432746,-0.104123,NaN
4,Aria Aber,/kaggle/input/datasets/srabongitikar/poetries/...,gpt2,SUCCESS,90,103,103,13,3.028895,2.654809,1.528094,1.297305,NaN
5,Arthur Sze,/kaggle/input/datasets/srabongitikar/poetries/...,gpt2,SUCCESS,1,1,1,1,1.719633,0.000000,NaN,NaN,NaN
6,Danez Smith,/kaggle/input/datasets/srabongitikar/poetries/...,gpt2,SUCCESS,15,58,58,9,1.732360,0.088256,-0.268445,2.926471,NaN
7,Don Paterson,/kaggle/input/datasets/srabongitikar/poetries/...,gpt2,SUCCESS,81,107,107,15,2.141105,1.308160,3.769707,14.517286,NaN
8,Eleanor Wilner,/kaggle/input/datasets/srabongitikar/poetries/...,gpt2,SUCCESS,110,132,132,18,1.714320,0.052926,2.224238,9.686500,NaN
9,Evie Shockley,/kaggle/input/datasets/srabongitikar/poetries/...,gpt2,SUCCESS,27,59,57,11,2.255177,0.734551,1.700289,2.349426,NaN


## Output files

The main file is `poet_source_moments_gpt2.csv`. It contains one row per target poet and the source-level `mean_bpc`, `variance_bpc`, `skewness_bpc`, and `excess_kurtosis_bpc`. Rows that could not be processed remain in the output with a status and missing moment values. Chunk scores and binned frequencies are exported separately for auditability.

In Kaggle, find the files under `/kaggle/working/poet_gpt2_moments/` and use **Save Version** with output saving enabled to download them.